In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from llama_cpp import Llama
from llama_cpp.llama_speculative import LlamaPromptLookupDecoding, LlamaDraftModel
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain.chat_models import init_chat_model
from openai import OpenAI


In [3]:
OPENAI_API_KEY = "YOUR_API_KEY"
import os

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


# llm = init_chat_model("ft:gpt-4o-mini-2024-07-18:tu-graz-hereditary:gutbrain-ie-finetune:B5qr9cGV", model_provider="openai")
llm = init_chat_model("gpt-4o-mini-2024-07-18", model_provider="openai")

In [4]:
model_path = "quants/llama-3-2-1B-instruct-lora.gguf"

In [5]:
model = Llama(
    model_path,
    n_gpu_layers=-1,
    n_ctx=4096,
    temperature=0.1,
    # draft_model=LlamaPromptLookupDecoding(num_pred_tokens=10),
)

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    yes
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GeForce RTX 4090, compute capability 8.9, VMM: yes
llama_load_model_from_file: using device CUDA0 (NVIDIA GeForce RTX 4090) - 23415 MiB free
llama_model_loader: loaded meta data with 29 key-value pairs and 147 tensors from quants/llama-3-2-1B-instruct-lora.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Epoch_4
llama_model_loader: - kv   3:                         general.size_label str              = 1.2B
llama_model_loader: - kv   4:                            general.license str

In [ ]:
from constrerl.annotator import (
    Annotator,
    Article,
    load_train,
    load_test,
    convert_to_enum_model,
    convert_to_string_model,
    article_to_enum_model,
    ExtendedEnumERLModel,
    ExtendedStringERLModel,
)
from constrerl._annotator_best import AnnotatorBest

In [23]:
data_path = "data/annotations/dev/dev.json"
out_path = "data/results/dev_out.json"
annotator = Annotator(langchain=llm, gen_tokens=2048, add_rag=True, reorder=False)
# annotator = Annotator(model=model, gen_tokens=2048)
eval_set = load_train(data_path)
few_shot_samples=10
annotator.add_prompt_examples([a for a in eval_set.values()][0:few_shot_samples])

In [24]:
with open("grammar.gbnf", "w") as f:
    f.write(annotator.erl_grammar)
print(annotator.erl_grammar)

root ::= (" "| "\n") grammar-models
grammar-models ::= relations
relations ::= "{"  ws "\"relations\"" ": " relations-relations  ws "}"
relation ::= "{"  ws "\"link_type\"" ": " relation-link-type ","  ws "\"subject_text_span\"" ": " string ","  ws "\"subject_location\"" ": " relation-subject-location ","  ws "\"object_text_span\"" ": " string ","  ws "\"object_location\"" ": " relation-object-location  ws "}"
relation-link-type ::= "\"anatomical location | located in | human\"" | "\"anatomical location | located in | animal\"" | "\"bacteria | interact | bacteria\"" | "\"bacteria | interact | chemical\"" | "\"bacteria | interact | drug\"" | "\"bacteria | influence | DDF\"" | "\"bacteria | change expression | gene\"" | "\"bacteria | located in | human\"" | "\"bacteria | located in | animal\"" | "\"bacteria | part of | microbiome\"" | "\"chemical | located in | anatomical location\"" | "\"chemical | located in | human\"" | "\"chemical | located in | animal\"" | "\"chemical | interact | c

In [25]:
annotator.example_messages

[[{'role': 'user',
   'content': "Title: Hypothesis of a potential BrainBiota and its relation to CNS autoimmune inflammation.\nAbstract: Infectious agents have been long considered to play a role in the pathogenesis of neurological diseases as part of the interaction between genetic susceptibility and the environment. The role of bacteria in CNS autoimmunity has also been highlighted by changes in the diversity of gut microbiota in patients with neurological diseases such as Parkinson's disease, Alzheimer disease and multiple sclerosis, emphasizing the role of the gut-brain axis. We discuss the hypothesis of a brain microbiota, the BrainBiota: bacteria living in symbiosis with brain cells. Existence of various bacteria in the human brain is suggested by morphological evidence, presence of bacterial proteins, metabolites, transcripts and mucosal-associated invariant T cells. Based on our data, we discuss the hypothesis that these bacteria are an integral part of brain development and i

In [26]:
test_samples = 10
annotations = annotator.annotate(
    {id: article.metadata for id, article in list(eval_set.items())[few_shot_samples:few_shot_samples+test_samples]}
)

Annotating articles: 100%|██████████| 10/10 [00:56<00:00,  5.66s/it, id=34480631]


In [27]:
annotations

{'30309367': Relations(relations=[Relation(link_type=<LinkType.DDF_affect_DDF: 'DDF | affect | DDF'>, subject_text_span='memory impairment', subject_location=<LabelLocation.ABSTRACT: 'abstract'>, object_text_span='Chronic inflammatory pain', object_location=<LabelLocation.ABSTRACT: 'abstract'>), Relation(link_type=<LinkType.chemical_interact_chemical: 'chemical | interact | chemical'>, subject_text_span="complete Freund's adjuvant (CFA)", subject_location=<LabelLocation.ABSTRACT: 'abstract'>, object_text_span='mice', object_location=<LabelLocation.ABSTRACT: 'abstract'>), Relation(link_type=<LinkType.microbiome_used_by_biomedical_technique: 'microbiome | used by | biomedical technique'>, subject_text_span='gut microbiota', subject_location=<LabelLocation.ABSTRACT: 'abstract'>, object_text_span='16S rRNA analysis', object_location=<LabelLocation.ABSTRACT: 'abstract'>), Relation(link_type=<LinkType.DDF_target_animal: 'DDF | target | animal'>, subject_text_span='spatial working memory impa

In [28]:
import numpy as np

print(
    np.mean(np.array([len(eval_set[k].relations) for k in annotations.keys()])),
    np.mean(np.array([len(a.relations) for a in annotations.values()])),
)

20.1 10.5


In [30]:
def f1k_annotations(y_true: list[str], y_pred: list[str], k: int = None):
    rel_set = set(y_true)
    # print(rel_set)
    doc_set = set(y_pred[:k])
    tp = len(doc_set.intersection(rel_set))  # docs that are in both -relevant docs
    fp = len(
        doc_set.difference(rel_set)
    )  # docs that are not in relevant set - irrelevant docs (false positiv)
    fn = len(
        rel_set.difference(doc_set)
    )  # relevant docs that are not present in doc set - missing docs
    if tp == 0:
        return 0, 0, 0
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    return 2 * precision * recall / (precision + recall), precision, recall


f1k_annotations_scores: list[float] = []

for id, article in annotations.items():
    gt_article = article_to_enum_model(eval_set[id], ExtendedEnumERLModel)
    gt_article = convert_to_string_model(gt_article, ExtendedStringERLModel)
    # print(article.relations[0].model_dump_json())
    y_true = [str(r.link_type) for r in gt_article.relations]
    y_pred = [str(r.link_type) for r in article.relations]
    print(y_true)
    print(y_pred)
    scores = f1k_annotations(y_true, y_pred)
    print(scores)
    f1k_annotations_scores.append(scores[0])

np.mean(f1k_annotations_scores)

['LinkType.DDF_strike_anatomical_location', 'LinkType.dietary_supplement_influence_DDF', 'LinkType.microbiome_is_linked_to_DDF', 'LinkType.microbiome_is_linked_to_DDF', 'LinkType.dietary_supplement_influence_DDF', 'LinkType.DDF_affect_DDF', 'LinkType.DDF_affect_DDF', 'LinkType.dietary_supplement_influence_DDF', 'LinkType.microbiome_is_linked_to_DDF', 'LinkType.microbiome_is_linked_to_DDF', 'LinkType.microbiome_is_linked_to_DDF', 'LinkType.DDF_strike_anatomical_location', 'LinkType.dietary_supplement_influence_DDF', 'LinkType.microbiome_is_linked_to_DDF', 'LinkType.DDF_affect_DDF', 'LinkType.microbiome_is_linked_to_DDF', 'LinkType.microbiome_is_linked_to_DDF']
['LinkType.DDF_affect_DDF', 'LinkType.chemical_interact_chemical', 'LinkType.microbiome_used_by_biomedical_technique', 'LinkType.DDF_target_animal', 'LinkType.DDF_strike_anatomical_location', 'LinkType.DDF_target_animal', 'LinkType.DDF_target_animal', 'LinkType.DDF_target_animal', 'LinkType.DDF_strike_anatomical_location', 'LinkTy

np.float64(0.35332442067736186)